# FAQ & Troubleshooting

Common problems and how to fix them.

---
## My TMAP has disconnected / hanging nodes

**Symptom:** Some points float alone or in small detached groups, not
connected to the main tree.

**Why it happens:** The kNN graph has multiple connected components.
This means some points couldn't find neighbors close enough to bridge
to the rest of the data. The MST of a disconnected graph is a forest
(multiple trees), and the detached trees appear as isolated groups.

**Fixes (try in order):**

1. **Increase `kc`** (candidate multiplier). This makes the LSH search
   try harder to find neighbors. Default is 10; try 50 or 100.

2. **Increase `n_neighbors`**. More neighbors means denser graph, more
   likely to be connected. Default is 10; try 20 or 30.

3. **Increase `n_permutations`** (Jaccard metric only). More MinHash
   permutations give more accurate similarity estimation, which helps
   the LSH find true neighbors. Default is 512; try 1024.

4. **Check your data.** If your data genuinely has well-separated
   clusters with no overlap, a disconnected graph is *correct* behavior.
   TMAP won't force connections that don't exist.

In [ ]:
from tmap import TMAP
import numpy as np

# Example: increasing kc and n_neighbors for better connectivity
# model = TMAP(n_neighbors=30, kc=100, seed=42).fit(data)


---
## My TMAP is too compact / points overlap

**Symptom:** The tree is squeezed into a small area, branches are
hard to distinguish.

**Fix: Increase `node_size`** in the `LayoutConfig`. This controls the
repulsion between nodes in the force-directed layout. Larger values
push nodes further apart.

Default is `1/65` (~0.015). Try `1/20` or `1/10` for more spread.

In [ ]:
from tmap.layout import LayoutConfig

cfg = LayoutConfig()
cfg.node_size = 1/20  # Increase from default 1/65

# model = TMAP(n_neighbors=20, layout_config=cfg, seed=42).fit(data)


---
## My TMAP is too spread out / branches are too long

**Symptom:** The tree takes up too much space, branches stretch out
with lots of empty whitespace.

**Fix: Decrease `node_size`**. Smaller values reduce repulsion.
Try `1/100` or even `1/200`.

In [ ]:
cfg = LayoutConfig()
cfg.node_size = 1/100  # Decrease for tighter layout

# model = TMAP(n_neighbors=20, layout_config=cfg, seed=42).fit(data)


---
## My TMAP looks different every time I run it

**Symptom:** Running the same code twice produces different layouts.

**Fix:** Set both `seed` and `deterministic` mode:

- `TMAP(seed=42)` seeds the layout algorithm and MinHash encoding.
- For full determinism, also set `cfg.deterministic = True` in the
  `LayoutConfig` — this forces single-threaded OGDF execution.

In [ ]:
cfg = LayoutConfig()
cfg.deterministic = True
cfg.seed = 42

# model = TMAP(n_neighbors=20, seed=42, layout_config=cfg).fit(data)
# Running this again will produce exactly the same output


---
## TMAP is slow. How do I speed it up?

The pipeline has three main bottlenecks. Here's how to address each:

**1. Encoding (Jaccard metric only)**
- Reduce `n_permutations`. 128 is often enough (default is 512).
- Ensure Numba is installed — it's 50-100x faster than the fallback.

**2. kNN search**
- For Jaccard: reduce `kc`. Default is 10; try 5 for faster (but less
  accurate) neighbor search.
- For cosine/euclidean: FAISS auto-selects HNSW for n >= 50k, which is
  much faster than brute-force.

**3. Layout**
- Reduce `layout_iterations`. Default is 1000; try 500 for faster
  convergence (slightly lower quality).
- Disable deterministic mode if you don't need reproducibility
  (multi-threaded OGDF is faster).

In [ ]:
# Fast settings for large datasets
# model = TMAP(
#     n_neighbors=15,
#     n_permutations=128,   # Faster encoding (Jaccard only)
#     kc=5,                 # Fewer candidates
#     layout_iterations=500,
#     seed=42,
# ).fit(data)


---
## "ImportError: metric='cosine' requires faiss"

Cosine and euclidean metrics use FAISS for nearest neighbor search.

Install it with:
```bash
pip install faiss-cpu
```

For GPU acceleration:
```bash
pip install faiss-gpu
```

If you can't install FAISS, compute the kNN graph yourself and pass
it via `fit(knn_graph=...)`. See the [Metric Guide](06_metric_guide.ipynb).

---
## Edges don't show in Jupyter notebook

The jupyter-scatter widget does not currently support edge rendering.
You'll see a warning: `"Edges are not supported in notebook mode yet"`.

**Workarounds:**
- Use `model.to_html("output.html")` — HTML export renders edges.
- Use `model.plot_static(edges=model.tree_)` — matplotlib draws edges.
- The tree topology is still accessible via `model.tree_` for
  programmatic use (paths, distances, etc.).

---
## LayoutConfig reference

All attributes of `LayoutConfig` (from the OGDF C++ extension):

| Attribute | Default | Effect |
|-----------|---------|--------|
| `k` | 10 | Neighbors per point (for `layout_from_lsh_forest`) |
| `kc` | 10 | Candidate multiplier for LSH queries |
| `fme_iterations` | 1000 | Force-directed layout iterations |
| `fme_precision` | 4 | Multipole expansion precision |
| `node_size` | 1/65 | Node repulsion radius |
| `mmm_repeats` | 1 | Multilevel mixer repeats |
| `sl_extra_scaling_steps` | 2 | Extra scaling refinement steps |
| `sl_scaling_type` | `RelativeToDrawing` | How scaling is computed |
| `sl_scaling_min` | 1.0 | Minimum scaling factor |
| `sl_scaling_max` | 1.0 | Maximum scaling factor |
| `sl_repeats` | 1 | Scaling layout repeats |
| `placer` | `Barycenter` | Node placement during uncoarsening |
| `merger` | `LocalBiconnected` | Coarsening strategy |
| `merger_factor` | 2.0 | Merger aggressiveness |
| `merger_adjustment` | 0 | Edge length adjustment |
| `deterministic` | False | Single-threaded for reproducibility |
| `seed` | None | Random seed (None = unseeded) |

**Most common adjustments:**
- `node_size` — controls spacing between points
- `fme_iterations` — more iterations = better quality but slower
- `deterministic` + `seed` — reproducible results

---
## TMAP estimator parameter reference

| Parameter | Default | Description |
|-----------|---------|-------------|
| `n_neighbors` | 10 | Number of nearest neighbors for kNN graph |
| `metric` | `'jaccard'` | `'jaccard'`, `'cosine'`, `'euclidean'`, or `'precomputed'` |
| `n_permutations` | 512 | MinHash signature length (Jaccard only) |
| `kc` | 10 | LSH candidate multiplier (Jaccard only) |
| `seed` | 42 | Random seed for layout and encoding |
| `minhash_seed` | 42 | Separate seed for MinHash permutations |
| `layout_iterations` | 1000 | Force-directed layout iterations |
| `layout_config` | None | Custom `LayoutConfig` for fine-grained control |
| `store_index` | False | Keep the ANN index for later `add_points()` |